<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Notebook 06</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Voice patient registration, confirmed by OTP</h1>
<p style="margin:.55em 0">Notebook 05's tools only <i>read</i>. This one <b>writes</b>: a front desk that registers a new patient by voice.
A wrong write is worse than a wrong answer, so the flow has two locks:</p>
<div style="display:flex;flex-direction:column;align-items:flex-start;gap:4px;margin:.8em 0">
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>patient speaks</b> name, phone, sex, date of birth (over one or more turns)</div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:2px solid #6366f1;background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>start_registration</b> (our code): validate → duplicate check → text a 6-digit code to the phone<br>
    <span style="opacity:.7;font-size:.9em">the model gets a read-back, <b>never the code</b></span></div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>model reads back</b> spelled names and the phone digit by digit, asks for the code</div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:2px solid #6366f1;background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>confirm_registration(code)</b> (our code): right code, within 5 min, 3 tries → write the record</div>
</div>
<p style="margin:.55em 0">The read-back catches <b>mishearing</b>. The code proves the <b>phone belongs to the patient</b>.</p>
<div style="margin:.8em 0;padding:.4em 1em;border-left:4px solid #6366f1;background:rgba(99,102,241,.12);border-radius:0 6px 6px 0">Simulated: a local SQLite test database (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">data/test.db</code>) and an outbox instead of SMS.
The rules are copied from DocAssist's registration (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">live/patients.py</code> names the source); nothing touches DocAssist.</div>

</div>

In [1]:
import contextlib
import json
import sqlite3

from live import audio, patients, realtime, tools

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">1. The test database</h2>
<p style="margin:.55em 0">Two fake patients: an adult and her child. They share a phone, which DocAssist allows for children.</p>

</div>

In [2]:
db = patients.init_db(reset=True)
db

WindowsPath('D:/Code/realtime-talk/data/test.db')

In [3]:
def show_patients():
    with contextlib.closing(sqlite3.connect(db)) as c:
        return c.execute("SELECT Patient_ID, First_Name, Last_Name, DOB, age, sex, Phone_Number FROM Patient").fetchall()

show_patients()

[(1, 'asha', 'sharma', '1985-03-02', 41, 'Female', '919812345678'),
 (2, 'rohan', 'mehta', '2015-07-19', 11, 'Male', '919812345678')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">2. DocAssist's rules, as plain functions</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">validate</code> returns a clean record or a list of problems. The phone gets the <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">91</code> prefix,
names are stored lower-case, and a date of birth is accepted as DD/MM/YYYY or YYYY-MM-DD:</p>

</div>

In [4]:
patients.validate("Ravi", "Kumar", "98765 43210", "Male", dob="15/08/1990")

({'first_name': 'ravi',
  'last_name': 'kumar',
  'phone': '919876543210',
  'sex': 'Male',
  'dob': datetime.date(1990, 8, 15),
  'age': 36,
  'email': None},
 [])

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Everything a misheard sentence might produce comes back as data, not a crash:</p>

</div>

In [5]:
patients.validate("R2", "Kumar", "12345", "M", dob="1990-08-15", age="50")[1]

['first name: letters only, 1-50 characters',
 'phone: must be a 10-digit mobile number (or 12 digits starting with 91)',
 'sex: must be Male or Female',
 "age 50 doesn't match date of birth 15 August 1990 (that is 36)"]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">3. The OTP lock, without the model</h2>
<p style="margin:.55em 0">A <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Registrar</code> is one front-desk conversation. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">start_registration</code> validates, checks duplicates,
and "texts" a code:</p>

</div>

In [6]:
reg = patients.Registrar(db)
out = reg.start_registration("Ravi", "Kumar", "9876543210", "Male", dob="1990-08-15")
out

{'status': 'code_sent',
 'sent_to': 'phone ending 3210',
 'read_back': {'name': 'Ravi Kumar',
  'spelled': 'R-A-V-I  K-U-M-A-R',
  'phone_digits': '9 8 7 6 5 4 3 2 1 0',
  'sex': 'Male',
  'date_of_birth': '15 August 1990',
  'age': 36},
 'next': 'First read back every detail in read_back: spell both names letter by letter and say the phone digits one by one. Then ask for the 6-digit code that was just texted.'}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">That dict is all the model will ever see. The code went to the phone, that is, the outbox:</p>

</div>

In [7]:
reg.outbox[-1]

{'to': '+919876543210', 'text': 'Your DocAssist registration code is 953124.'}

In [8]:
code = reg.outbox[-1]["text"][-7:-1]
code in json.dumps(out)   # is the code anywhere in the model's view?

False

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">A wrong code counts as a try. The right one writes the record, once:</p>

</div>

In [9]:
reg.confirm_registration("000000")

{'status': 'wrong_code', 'tries_left': 2}

In [10]:
reg.confirm_registration(code)

{'status': 'registered', 'patient_id': 3, 'name': 'Ravi Kumar'}

In [11]:
reg.confirm_registration(code)   # single use

{'status': 'error',
 'problem': 'no registration is waiting for a code; collect the details first'}

In [12]:
show_patients()

[(1, 'asha', 'sharma', '1985-03-02', 41, 'Female', '919812345678'),
 (2, 'rohan', 'mehta', '2015-07-19', 11, 'Male', '919812345678'),
 (3, 'ravi', 'kumar', '1990-08-15', 36, 'Male', '919876543210')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">4. Duplicates</h2>
<p style="margin:.55em 0">An adult's phone must be unique. A child may share it:</p>

</div>

In [13]:
reg.start_registration("Kavya", "Rao", "9812345678", "Female", age="34")

{'status': 'invalid',
 'problems': ['this phone is already registered to an adult patient (Asha S.)']}

In [14]:
reg.start_registration("Mini", "Sharma", "9812345678", "Female", age="6")["status"]

'code_sent'

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">5. Tools and instructions</h2>
<p style="margin:.55em 0">The two methods become tools (same <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">live.tools</code> registry as notebook 05). Optional fields are strings,
and <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">""</code> means not given. The instructions set the order: collect, read back, ask for the code, confirm.</p>

</div>

In [15]:
INSTRUCTIONS = """You are the front desk of a hospital, registering new patients by voice. Be brief and warm.
Collect: first name, last name, 10-digit mobile number, sex (Male or Female), and date of birth (or age if they do not know it).
Ask only for what is missing. Do not ask for email.
As soon as you have all of them, call start_registration right away. Do not read the details back before calling it:
its result is what you read back. Spell the names and say the phone number digit by digit.
Then ask for the 6-digit code that was just texted to their phone.
If they correct anything, call start_registration again with the fixed details; a new code is sent.
When they say the code, call confirm_registration with the digits.
Only say the patient is registered after confirm_registration returns status registered, then give the patient ID."""

In [16]:
reg = patients.Registrar(db)
REGISTRY = reg.tools(tools.spec)
SPECS = [s for s, _ in REGISTRY.values()]
[s['name'] for s in SPECS]

['start_registration', 'confirm_registration']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">6. One registration in text first</h2>
<p style="margin:.55em 0">Text in (audio still comes out) makes the protocol easy to follow. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">say_text</code> runs any tool calls and
keeps going until the model answers in words:</p>

</div>

In [17]:
ws = await realtime.start_session(INSTRUCTIONS, tools=SPECS, max_output_tokens=1000)

async def say_text(text):
    await realtime.send(ws, realtime.user_text(text))
    await realtime.send(ws, {"type": "response.create"})
    while True:
        ev = await realtime.collect(ws, until="response.done")
        handled = await realtime.answer_tool_calls(ws, ev[-1]["response"], REGISTRY)
        for name, args, out in handled:
            print("TOOL", name, " ".join(args.split()), "->", json.dumps(out))
        if not handled:
            return [e["transcript"] for e in ev if e["type"] == "response.output_audio_transcript.done"]

In [18]:
await say_text("Hi, I would like to register. I am Priya Nair, mobile 9123456780, female, born 2nd March 1992.")

TOOL start_registration { "first_name": "Priya", "last_name": "Nair", "phone": "9123456780", "sex": "Female", "dob": "1992-03-02", "age": "", "email": "" } -> {"result": {"status": "code_sent", "sent_to": "phone ending 6780", "read_back": {"name": "Priya Nair", "spelled": "P-R-I-Y-A  N-A-I-R", "phone_digits": "9 1 2 3 4 5 6 7 8 0", "sex": "Female", "date_of_birth": "02 March 1992", "age": 34}, "next": "First read back every detail in read_back: spell both names letter by letter and say the phone digits one by one. Then ask for the 6-digit code that was just texted."}}


["Here’s what we have:\n\nName: Priya Nair, spelled P-R-I-Y-A, then N-A-I-R. The phone number is 9 1 2 3 4 5 6 7 8 0. The sex is Female, and the date of birth is 2nd March 1992.\n\nNow, please check and make sure everything is correct. Once you're sure, let me know the 6-digit code that was texted to your phone."]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The patient reads the code from their phone:</p>

</div>

In [19]:
code = reg.outbox[-1]["text"][-7:-1]
code

'740689'

In [20]:
await say_text(f"Yes, that is right. The code is {code}.")

TOOL confirm_registration { "code": "740689" } -> {"result": {"status": "registered", "patient_id": 4, "name": "Priya Nair"}}


['You are now registered. Your patient ID is 4. If you need any further assistance, let me know!']

In [21]:
await ws.close()
show_patients()

[(1, 'asha', 'sharma', '1985-03-02', 41, 'Female', '919812345678'),
 (2, 'rohan', 'mehta', '2015-07-19', 11, 'Male', '919812345678'),
 (3, 'ravi', 'kumar', '1990-08-15', 36, 'Male', '919876543210'),
 (4, 'priya', 'nair', '1992-03-02', 34, 'Female', '919123456780')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">7. By voice: the happy path</h2>
<p style="margin:.55em 0">A <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">ScriptedUser</code> (in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">live/audio.py</code>) speaks each line only after the model's answer has
played out. A line can be a function: the code line reads the outbox at the moment it is spoken, like a person
looking at their phone.</p>

</div>

In [22]:
spk = audio.Speaker(); spk.stream.start()

def otp_line(prefix=""):
    code = reg.outbox[-1]["text"][-7:-1]
    return f"{prefix}The code is {', '.join(code)}."

async def voice_run(lines, seconds=60):
    global reg  # a fresh front-desk conversation per run
    reg = patients.Registrar(db)
    user = audio.ScriptedUser(lines, spk)
    ws = await realtime.start_session(INSTRUCTIONS, tools=SPECS, max_output_tokens=1000)
    try:
        log = await realtime.converse(ws, user.chunks(), spk, seconds=seconds, tools=reg.tools(tools.spec),
                                      on_note=user.on_note, stop=user.done)
    finally:
        await ws.close()
    return user.said, log

def story(log):
    return [row for row in log if row[1] in ("user_said", "tool_call", "filler", "model_said")]

In [23]:
said, log = await voice_run([
    "Hi, I would like to register as a new patient. My name is Arjun Patel. "
    "My mobile is 9 8 7 0 0 1 2 3 4 5. I am male, born on the 5th of June 1988.",
    lambda: otp_line("Yes, that is all correct. "),
])

0.87 input_audio_buffer.speech_started 0.00s of audio still queued


4.56 input_audio_buffer.speech_stopped 3552
4.56 input_audio_buffer.committed 
4.57 response.created in_progress


5.09 first_audio_delta 
5.09 input_audio_buffer.speech_started 0.40s of audio still queued
5.09 barge_in dropped 0.40s, truncate at 0 ms
5.09 model_said Of course! Let's start with
5.09 response.done cancelled
5.09 user_said Hi, I would like to register as a new patient.


5.42 conversation.item.truncated 0


7.06 input_audio_buffer.speech_stopped 5824
7.06 input_audio_buffer.committed 
7.07 response.created in_progress


7.44 first_audio_delta 
7.44 input_audio_buffer.speech_started 0.40s of audio still queued
7.44 barge_in dropped 0.40s, truncate at 0 ms
7.44 model_said Thank you, Arjun Patel.
7.44 response.done cancelled
7.45 user_said My name is Arjun Patel.


... (13 outputs left out)

51.13 filler "Let me check." while confirm_registration runs
51.19 response.done completed
51.28 tool_call confirm_registration({ "code": "939769" }) -> {"result": {"status": "registered", "patient_id": 5, "name": "Arjun Patel"}}


51.35 user_said The code is 9-3-9-7-6-9.


51.67 response.created in_progress


51.94 first_audio_delta 


In [24]:
said

['Hi, I would like to register as a new patient. My name is Arjun Patel. My mobile is 9 8 7 0 0 1 2 3 4 5. I am male, born on the 5th of June 1988.',
 'Yes, that is all correct. The code is 9, 3, 9, 7, 6, 9.']

In [25]:
story(log)

[(5.09, 'model_said', "Of course! Let's start with"),
 (5.09, 'user_said', 'Hi, I would like to register as a new patient.'),
 (7.44, 'model_said', 'Thank you, Arjun Patel.'),
 (7.45, 'user_said', 'My name is Arjun Patel.'),
 (12.35, 'user_said', 'My mobile is 9870012345.'),
 (16.7, 'filler', '"Let me check." while start_registration runs'),
 (16.93,
  'user_said',
  'I am male, born on the fifth of June nineteen eighty-eight.'),
 (17.03,
  'tool_call',
  'start_registration({ "first_name": "Arjun", "last_name": "Patel", "phone": "9870012345", "sex": "Male", "dob": "1988-06-05", "age": "", "email": "" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2345", "read_back": {"name": "Arjun Patel", "spelled": "A-R-J-U-N  P-A-T-E-L", "phone_digits": "9 8 7 0 0 1 2 3 4 5", "sex": "Male", "date_of_birth": "05 June 1988", "age": 38}, "next": "First read back every detail in read_back: spell both names letter by letter and say the phone digits one by one. Then ask for the 6-digit

In [26]:
show_patients()

[(1, 'asha', 'sharma', '1985-03-02', 41, 'Female', '919812345678'),
 (2, 'rohan', 'mehta', '2015-07-19', 11, 'Male', '919812345678'),
 (3, 'ravi', 'kumar', '1990-08-15', 36, 'Male', '919876543210'),
 (4, 'priya', 'nair', '1992-03-02', 34, 'Female', '919123456780'),
 (5, 'arjun', 'patel', '1988-06-05', 38, 'Male', '919870012345')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8. A duplicate phone</h2>
<p style="margin:.55em 0">Kavya gives the phone of the adult already on file. No code may be sent, and the model must say why:</p>

</div>

In [27]:
said, log = await voice_run([
    "Hello, please register me. I am Kavya Rao, female, 34 years old, mobile 9 8 1 2 3 4 5 6 7 8.",
])
story(log)

0.83 input_audio_buffer.speech_started 0.00s of audio still queued


3.43 input_audio_buffer.speech_stopped 2560
3.43 input_audio_buffer.committed 
3.44 response.created in_progress


3.91 first_audio_delta 
3.91 user_said Hello, please register me.
4.03 input_audio_buffer.speech_started 0.70s of audio still queued
4.03 barge_in dropped 0.70s, truncate at 104 ms
4.03 model_said Of course! Let's get started. Could you please
4.03 response.done cancelled


4.33 conversation.item.truncated 104


12.05 input_audio_buffer.speech_stopped 10432
12.05 input_audio_buffer.committed 
12.06 response.created in_progress


12.38 filler "Let me check." while start_registration runs
12.49 user_said I am Kavya Rao, female, 34 years old, mobile 9812345678.


12.63 response.done completed
12.64 tool_call start_registration({ "first_name": "Kavya", "last_name": "Rao", "phone": "9812345678", "sex": "Female", "dob": "", "age": "34", "email": "" }) -> {"result": {"status": "invalid", "problems": ["this phone is already registered to an adult patient (Asha S.)"]}}


13.2 response.created in_progress


13.56 first_audio_delta 


14.6 model_said It seems that the phone number you provided is already registered. Could you double-check the number or provide a different contact? If the number is correct, we might need to investigate further.
14.6 response.done completed


[(3.91, 'user_said', 'Hello, please register me.'),
 (4.03, 'model_said', "Of course! Let's get started. Could you please"),
 (12.38, 'filler', '"Let me check." while start_registration runs'),
 (12.49,
  'user_said',
  'I am Kavya Rao, female, 34 years old, mobile 9812345678.'),
 (12.64,
  'tool_call',
  'start_registration({ "first_name": "Kavya", "last_name": "Rao", "phone": "9812345678", "sex": "Female", "dob": "", "age": "34", "email": "" }) -> {"result": {"status": "invalid", "problems": ["this phone is already registered to an adult patient (Asha S.)"]}}'),
 (14.6,
  'model_said',
  'It seems that the phone number you provided is already registered. Could you double-check the number or provide a different contact? If the number is correct, we might need to investigate further.')]

In [28]:
reg.outbox   # nothing was texted

[]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">9. A correction before confirming</h2>
<p style="margin:.55em 0">After the read-back the patient fixes the spelling. The model calls <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">start_registration</code> again, a
<b>new</b> code is texted, and only the new one works:</p>

</div>

In [29]:
said, log = await voice_run([
    "Hi, I want to register. Neha Verma, female, born 12 January 2000, mobile 9 1 1 1 2 2 2 3 3 3.",
    "Sorry, my last name is Varma, V A R M A.",
    lambda: otp_line("Yes, now it is right. "),
], seconds=90)
story(log)

0.79 input_audio_buffer.speech_started 0.00s of audio still queued


3.3 input_audio_buffer.speech_stopped 2432
3.3 input_audio_buffer.committed 
3.3 response.created in_progress


3.61 input_audio_buffer.speech_started 0.00s of audio still queued
3.64 model_said Of course!
3.72 response.done cancelled
3.72 user_said Hi, I want to register.


12.62 input_audio_buffer.speech_stopped 11040
12.62 input_audio_buffer.committed 
12.62 response.created in_progress


12.91 filler "Let me check." while start_registration runs
13.06 user_said Neha Verma, female, born January 12, 2000, mobile 91112223333.


13.23 response.done completed
13.23 tool_call start_registration({ "first_name": "Neha", "last_name": "Verma", "phone": "9111222333", "sex": "Female", "dob": "2000-01-12", "age": "", "email": "" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2333", "read_back": {"name": "Neha Verma", "spelled": "N-E-H-A  V-E-R-M-A", "phone_digits": "9 1 1 1 2 2 2 3 3 3", "sex": "Female", "date_of_birth": "12 January 2000", "age": 26}, "next": "First read back every detail in read_back: spell both names letter by letter and say the phone digits one by one. Then ask for the 6-digit code that was just texted."}}


... (15 outputs left out)

73.93 response.created in_progress


74.27 first_audio_delta 


74.91 model_said You're all set! Neha Varma is now registered. Your patient ID is 6. If you need anything else, let me know.
74.92 response.done completed


[(3.64, 'model_said', 'Of course!'),
 (3.72, 'user_said', 'Hi, I want to register.'),
 (12.91, 'filler', '"Let me check." while start_registration runs'),
 (13.06,
  'user_said',
  'Neha Verma, female, born January 12, 2000, mobile 91112223333.'),
 (13.23,
  'tool_call',
  'start_registration({ "first_name": "Neha", "last_name": "Verma", "phone": "9111222333", "sex": "Female", "dob": "2000-01-12", "age": "", "email": "" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2333", "read_back": {"name": "Neha Verma", "spelled": "N-E-H-A  V-E-R-M-A", "phone_digits": "9 1 1 1 2 2 2 3 3 3", "sex": "Female", "date_of_birth": "12 January 2000", "age": 26}, "next": "First read back every detail in read_back: spell both names letter by letter and say the phone digits one by one. Then ask for the 6-digit code that was just texted."}}'),
 (16.17,
  'model_said',
  'Thank you! Here’s what we have:\n\nFirst name: Neha (spelled N-E-H-A)\nLast name: Verma (spelled V-E-R-M-A)\nPhone number

In [30]:
[m["text"] for m in reg.outbox]

['Your DocAssist registration code is 295445.',
 'Your DocAssist registration code is 959025.']

In [31]:
show_patients()

[(1, 'asha', 'sharma', '1985-03-02', 41, 'Female', '919812345678'),
 (2, 'rohan', 'mehta', '2015-07-19', 11, 'Male', '919812345678'),
 (3, 'ravi', 'kumar', '1990-08-15', 36, 'Male', '919876543210'),
 (4, 'priya', 'nair', '1992-03-02', 34, 'Female', '919123456780'),
 (5, 'arjun', 'patel', '1988-06-05', 38, 'Male', '919870012345'),
 (6, 'neha', 'varma', '2000-01-12', 26, 'Female', '919111222333')]

In [32]:
spk.stream.close()

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Takeaways</span>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">A write needs two locks: a <b>read-back</b> (spelled names, digits one by one) against mishearing, and an
      <b>OTP</b> against registering someone else's phone.</li>
  <li style="margin:.2em 0">The code never enters the model's context: our code texts it and our code checks it.
      The model only carries the digits the patient says.</li>
  <li style="margin:.2em 0">Corrections are just another <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">start_registration</code> call. The old code dies with the old details.</li>
</ul>
<p style="margin:.55em 0"><b>For real DocAssist:</b> swap SQLite for its registration service, the outbox for its SMS/WhatsApp sender, and
decide whether patient details may go to OpenAI. <b>Next:</b> this flow in the web app, with the form filling in as the patient talks.</p>

</div>